In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../FMRs_Excepciones_27.08.26_Oeste.xlsx")

In [3]:
df_normalizado = pd.read_excel("../FMRs_Excepciones_27.08.26_Oeste.xlsx", sheet_name="FMR_Normalizado")

In [4]:
print(df_normalizado.shape)
print(df_normalizado.columns.tolist())

(301, 13)
['SubSistema', 'FMR', 'PO', 'LINE', 'DESCRIPCION', 'STATUS', 'STATUS_2', 'COMENTARIOS', 'BUYER', 'EXPEDITOR', 'ETA', 'ON SITE', 'ENTREGADO']


In [5]:
#df_normalizado.head(20)

In [6]:
df_normalizado.info()

<class 'pandas.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   SubSistema   294 non-null    str   
 1   FMR          278 non-null    object
 2   PO           146 non-null    object
 3   LINE         61 non-null     str   
 4   DESCRIPCION  217 non-null    str   
 5   STATUS       222 non-null    str   
 6   STATUS_2     180 non-null    str   
 7   COMENTARIOS  57 non-null     str   
 8   BUYER        222 non-null    str   
 9   EXPEDITOR    112 non-null    str   
 10  ETA          111 non-null    object
 11  ON SITE      67 non-null     str   
 12  ENTREGADO    30 non-null     str   
dtypes: object(3), str(10)
memory usage: 30.7+ KB


In [7]:
print("SubSistemas únicos:", df_normalizado["SubSistema"].nunique())
print("FMR únicas:", df_normalizado["FMR"].nunique())
print("Filas:", len(df_normalizado))

SubSistemas únicos: 42
FMR únicas: 34
Filas: 301


In [8]:
# REVISION DE DUPLICADOS
duplicados = df_normalizado.duplicated(
    subset=["SubSistema", "FMR"]
).sum()

print("Combinaciones SubSistema + FMR duplicadas:", duplicados)

Combinaciones SubSistema + FMR duplicadas: 159


1. ANALISIS DE STATUS

In [9]:
df_normalizado["STATUS"].value_counts()

STATUS
k. Awarded                      120
b. RFQ Issued                    49
f. Bid Tab Issued to MYSRL       24
c. TEA Requested                 19
h. FPO to Supplier Signature      7
In Process                        2
a. FMR Received                   1
Name: count, dtype: int64

2. ANALISIS ON SITE

In [10]:
df_normalizado["ON SITE"].value_counts(dropna=False)

ON SITE
NaN        234
SI          33
NO          29
PARCIAL      5
Name: count, dtype: int64

3. ANALISIS ENTREGADO A CONSTRUCCION

In [11]:
df_normalizado["ENTREGADO"].value_counts(dropna=False)

ENTREGADO
NaN        271
NO          24
PARCIAL      3
SI           3
Name: count, dtype: int64

4. CREACIÓN DE TABLA PUENTE ENTRE df y df_normalizado

In [12]:
df_subsistema_fmr = (
    df_normalizado[["SubSistema", "FMR"]]
    .dropna(subset=["SubSistema", "FMR"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(df_subsistema_fmr.shape)

(133, 2)


In [13]:
print("SubSistemas únicos:", df_subsistema_fmr["SubSistema"].nunique())
print("FMR únicas:", df_subsistema_fmr["FMR"].nunique())

df_subsistema_fmr.head(20)

SubSistemas únicos: 42
FMR únicas: 33


,SubSistema,FMR
0,36045-ECM-002,739023
1,36045-ECM-005,739023
2,36045-ECM-008,739023
3,36045-ECM-010,748849
4,36045-NS-002,748849
5,36045-NS-003,748849
6,36045-ER-002,758825
7,36045-ECM-005,758825
8,36045-ECM-011,758825
9,36045-ECM-018,762079


In [14]:
# Verificación de duplicados:
duplicados = df_subsistema_fmr.duplicated(
    subset=["SubSistema", "FMR"]
).sum()

print("Duplicados:", duplicados)

Duplicados: 0


In [15]:
# Comprobar relación muchos a muchos - ¿A cuántos subsistemas pertenece cada FMR?
fmr_por_subsistema = (
    df_subsistema_fmr
    .groupby("FMR")["SubSistema"]
    .nunique()
    .sort_values(ascending=False)
)

print(fmr_por_subsistema.head(20))

FMR
747667    11
791795     9
730855     9
791970     9
785312     9
789066     7
781845     7
787641     7
787742     7
786400     6
750147     5
777905     5
748849     4
783394     4
748204     4
739023     3
750959     3
779569     3
758825     3
797282     2
Name: SubSistema, dtype: int64


In [16]:
# Comprobar relación muchos a muchos - ¿Cuántas FMR tiene cada SubSistema?
subsistema_por_fmr = (
    df_subsistema_fmr
    .groupby("SubSistema")["FMR"]
    .nunique()
    .sort_values(ascending=False)
)

print(subsistema_por_fmr.head(20))

SubSistema
36045-ROT-005     11
36045-ROT-003     11
36045-ROT-004     11
36045-NS-003      10
36045-ROT-007      9
36045-ROT-006      9
36045-NS-002       8
36045-ROT-002      8
36045-ROT-001      8
36045-AB-001       4
36045-JD1-001      4
36045-JD1-006      3
36045-JD1-005      3
36045-ECM-005      2
36045-VOD-001      2
36045-JDS-001      2
36045-FDS-003      2
36045-ECM-011      2
36045-ESHA-001     1
36045-ER-002       1
Name: FMR, dtype: int64


In [17]:
fmr_df = set(
    df["FMR"]
    .dropna()
    .astype(str)
    .str.strip()
)

fmr_normalizado = set(
    df_subsistema_fmr["FMR"]
    .dropna()
    .astype(str)
    .str.strip()
)

fmr_sin_subsistema = fmr_df - fmr_normalizado

print("FMR en df:", len(fmr_df))
print("FMR en FMR_Normalizado:", len(fmr_normalizado))
print("FMR sin SubSistema:", len(fmr_sin_subsistema))
print(fmr_sin_subsistema)

FMR en df: 35
FMR en FMR_Normalizado: 33
FMR sin SubSistema: 2
{'…..................', '788286'}


5. ANALISIS DE ESTRUCTURA DE df

In [18]:
fmr_po = (
    df[["FMR", "PO"]]
    .dropna(subset=["FMR"])
    .drop_duplicates()
)

print(fmr_po.shape)
print("FMR únicas:", fmr_po["FMR"].nunique())
print("PO únicas:", fmr_po["PO"].nunique())

(50, 2)
FMR únicas: 35
PO únicas: 33


In [19]:
po_por_fmr = (
    fmr_po
    .groupby("FMR")["PO"]
    .nunique()
    .sort_values(ascending=False)
)

print(po_por_fmr)

FMR
748849                 5
747667                 4
762079                 3
748204                 2
739023                 2
724411                 2
290863                 1
393928                 1
292230                 1
730855                 1
728366                 1
777905                 1
750959                 1
758825                 1
763898                 1
786400                 1
783399                 1
783394                 1
780263                 1
…..................    1
789066                 1
750147                 0
748236                 0
779569                 0
766793                 0
781845                 0
785312                 0
787733                 0
787641                 0
788286                 0
787742                 0
791795                 0
791970                 0
797282                 0
Express N°22?          0
Name: PO, dtype: int64


In [20]:
fmr_por_po = (
    fmr_po
    .groupby("PO")["FMR"]
    .nunique()
    .sort_values(ascending=False)
)

print(fmr_por_po)

PO
372068                    1
372529                    1
549934                    1
734498                    1
749089                    1
768319                    1
768320                    1
770196                    1
773448                    1
773449                    1
775430                    1
775431                    1
778686                    1
778687                    1
782496                    1
783308                    1
783376                    1
783377                    1
784422                    1
786062                    1
786063                    1
786064                    1
790214                    1
791926                    1
792338                    1
794129                    1
796220                    1
796237                    1
796255                    1
798745                    1
801426                    1
783376\n783377\n791926    1
….........                1
Name: FMR, dtype: int64


6. INVESTIGACION FMR SIN DATOS

In [21]:
df[df["FMR"].astype(str).str.strip() == "788286"]

,SubSistema,FMR,PO,LINE,DESCRIPCION,STATUS,STATUS_2,COMENTARIOS,BUYER,EXPEDITOR,ETA,ON SITE,ENTREGADO,Unnamed: 13
92,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
df_normalizado[
    df_normalizado["FMR"].astype(str).str.strip() == "788286"
]

,SubSistema,FMR,PO,LINE,DESCRIPCION,STATUS,STATUS_2,COMENTARIOS,BUYER,EXPEDITOR,ETA,ON SITE,ENTREGADO
263,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
265,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
267,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
269,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
271,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
273,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
275,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
df[df["FMR"].astype(str).str.strip() == "….................."]

,SubSistema,FMR,PO,LINE,DESCRIPCION,STATUS,STATUS_2,COMENTARIOS,BUYER,EXPEDITOR,ETA,ON SITE,ENTREGADO,Unnamed: 13
98,….....................,…..................,….........,…...........,….....................,….....................,….....................,….....................,….....................,….....................,….............,….............,….............,NaN


In [24]:
fmr_po = (
    df[["FMR", "PO"]]
    .dropna(subset=["FMR"])
    .drop_duplicates()
)

print(fmr_po.shape)
print("FMR únicas:", fmr_po["FMR"].nunique())
print("PO únicas:", fmr_po["PO"].nunique())

(50, 2)
FMR únicas: 35
PO únicas: 33


In [25]:
po_por_fmr = (
    fmr_po
    .groupby("FMR")["PO"]
    .nunique()
    .sort_values(ascending=False)
)

print(po_por_fmr)

FMR
748849                 5
747667                 4
762079                 3
748204                 2
739023                 2
724411                 2
290863                 1
393928                 1
292230                 1
730855                 1
728366                 1
777905                 1
750959                 1
758825                 1
763898                 1
786400                 1
783399                 1
783394                 1
780263                 1
…..................    1
789066                 1
750147                 0
748236                 0
779569                 0
766793                 0
781845                 0
785312                 0
787733                 0
787641                 0
788286                 0
787742                 0
791795                 0
791970                 0
797282                 0
Express N°22?          0
Name: PO, dtype: int64


In [26]:
fmr_por_po = (
    fmr_po
    .groupby("PO")["FMR"]
    .nunique()
    .sort_values(ascending=False)
)

print(fmr_por_po)

PO
372068                    1
372529                    1
549934                    1
734498                    1
749089                    1
768319                    1
768320                    1
770196                    1
773448                    1
773449                    1
775430                    1
775431                    1
778686                    1
778687                    1
782496                    1
783308                    1
783376                    1
783377                    1
784422                    1
786062                    1
786063                    1
786064                    1
790214                    1
791926                    1
792338                    1
794129                    1
796220                    1
796237                    1
796255                    1
798745                    1
801426                    1
783376\n783377\n791926    1
….........                1
Name: FMR, dtype: int64


In [27]:
# INVESTIGAR LAS DOS FMR SIN SUBSISTEMA
df[df["FMR"].astype(str).str.strip() == "788286"]

,SubSistema,FMR,PO,LINE,DESCRIPCION,STATUS,STATUS_2,COMENTARIOS,BUYER,EXPEDITOR,ETA,ON SITE,ENTREGADO,Unnamed: 13
92,NaN,788286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
df[df["FMR"].astype(str).str.strip() == "….................."]

,SubSistema,FMR,PO,LINE,DESCRIPCION,STATUS,STATUS_2,COMENTARIOS,BUYER,EXPEDITOR,ETA,ON SITE,ENTREGADO,Unnamed: 13
98,….....................,…..................,….........,…...........,….....................,….....................,….....................,….....................,….....................,….....................,….............,….............,….............,NaN


In [29]:
fmr_po = (
    df[["FMR", "PO"]]
    .dropna(subset=["FMR"])
    .drop_duplicates()
)

print(fmr_po.shape)
print("FMR únicas:", fmr_po["FMR"].nunique())
print("PO únicas:", fmr_po["PO"].nunique())

(50, 2)
FMR únicas: 35
PO únicas: 33


6. INSPECCIÓN DE PO

In [30]:
#df[df["PO"].notna()][["FMR", "PO"]].sort_values("FMR")
# ERROR AL CORRER EL CÓDIGO
#TypeError: '<' not supported between instances of 'str' and 'int'

In [31]:
po_multilinea = df[
    df["PO"].astype(str).str.contains("\n", regex=False, na=False)
][["FMR", "PO"]]

print(po_multilinea)

       FMR                      PO
55  747667  783376\n783377\n791926


In [32]:
po_con_separador = df[
    df["PO"].astype(str).str.contains("/", regex=False, na=False)
][["FMR", "PO"]]

print(po_con_separador)

Empty DataFrame
Columns: [FMR, PO]
Index: []


In [33]:
po_basura = df[
    df["PO"].astype(str).str.contains("…", regex=False, na=False)
][["FMR", "PO"]]

print(po_basura)

                    FMR          PO
98  …..................  ….........


7. REVISIÓN DE FILAS DE FMR CON PO MULTIPLES

In [34]:
fmr_multiples_po = [
    "748849",
    "747667",
    "762079",
    "748204",
    "739023",
    "724411"
]

df[
    df["FMR"].astype(str).str.strip().isin(fmr_multiples_po)
][[
    "FMR",
    "PO",
    "LINE",
    "DESCRIPCION",
    "STATUS",
    "STATUS_2",
    "BUYER",
    "EXPEDITOR",
    "ETA",
    "ON SITE",
    "ENTREGADO"
]]

,FMR,PO,LINE,DESCRIPCION,STATUS,STATUS_2,BUYER,EXPEDITOR,ETA,ON SITE,ENTREGADO
0,739023,768319,NaN,Material permanente - Terminales para la conex...,k. Awarded,Adjudicado,Alejandro Aracayo,NaN,2026-07-25 00:00:00,SI,NO
1,739023,768320,NaN,Material permanente - Terminales para la conex...,k. Awarded,Adjudicado,Alejandro Aracayo,NaN,2026-07-24 00:00:00,SI,NO
2,739023,768320,NaN,NaN,k. Awarded,Adjudicado,Alejandro Aracayo,NaN,07-Aug-2026,SI,NO
3,739023,768320,NaN,NaN,k. Awarded,Adjudicado,Alejandro Aracayo,NaN,20-Aug-2026,NO,NaN
4,739023,768320,NaN,NaN,k. Awarded,Adjudicado,Alejandro Aracayo,NaN,2026-09-28 00:00:00,NO,NaN
5,739023,NaN,NaN,Material permanente - Terminales para la conex...,f. Bid Tab Issued to MYSRL,Se envió el Bid Tab al cliente (MYSRL). Client...,Alejandro Aracayo,NaN,NaN,NaN,NaN
6,739023,NaN,Item 3,"Fuse Cover, PN-D172948",f. Bid Tab Issued to MYSRL,Se envió el Bid Tab al cliente (MYSRL). Client...,Alejandro Aracayo,NaN,NaN,NaN,NaN
7,739023,NaN,Item 6,Terminal lug compatible with 140G Circuit Brak...,f. Bid Tab Issued to MYSRL,Se envió el Bid Tab al cliente (MYSRL). Client...,Alejandro Aracayo,NaN,NaN,NaN,NaN
8,739023,NaN,Item 9,Terminal lug compatible with 140G Circuit Brak...,f. Bid Tab Issued to MYSRL,Se envió el Bid Tab al cliente (MYSRL). Client...,Alejandro Aracayo,NaN,NaN,NaN,NaN
9,739023,NaN,Item 10,Terminal lug compatible with 140G Circuit Brak...,f. Bid Tab Issued to MYSRL,Se envió el Bid Tab al cliente (MYSRL). Client...,Alejandro Aracayo,NaN,NaN,NaN,NaN


8. CORRECCIÓN DE TIPOS EN 'FMR' Y 'PO'

In [35]:
df_fmr_po = df[["FMR", "PO"]].copy()

df_fmr_po["FMR"] = (
    df_fmr_po["FMR"]
    .astype("string")
    .str.strip()
)

df_fmr_po["PO"] = (
    df_fmr_po["PO"]
    .astype("string")
    .str.strip()
)

9. ELIMINAMOS FMR BASURA O INCOMPLETAS

In [36]:
# Eliminación de filas donde no exista FMR
df_fmr_po = df_fmr_po.dropna(subset=["FMR"])

In [37]:
# Eliminamos valores basura
df_fmr_po = df_fmr_po[
    ~df_fmr_po["FMR"].str.contains("…", regex=False, na=False)
]

In [38]:
# Eliminamos filas sin PO:
df_fmr_po = df_fmr_po.dropna(subset=["PO"])

In [39]:
# SEPARAMOS LA PO DE 747667
df_fmr_po["PO"] = df_fmr_po["PO"].str.split("\n")

In [40]:
# EXPLODE
df_fmr_po = df_fmr_po.explode("PO")

In [41]:
# LIMPIEZA
df_fmr_po["PO"] = (
    df_fmr_po["PO"]
    .astype("string")
    .str.strip()
)

In [42]:
# Eliminamos PO Basura
df_fmr_po = df_fmr_po[
    ~df_fmr_po["PO"].str.contains("…", regex=False, na=False)
]

In [43]:
df_fmr_po = (
    df_fmr_po
    .dropna(subset=["FMR", "PO"])
    .drop_duplicates()
    .reset_index(drop=True)
)

In [44]:
# Verificación de resultados
print(df_fmr_po.shape)

print("FMR únicas:", df_fmr_po["FMR"].nunique())
print("PO únicas:", df_fmr_po["PO"].nunique())

print(
    "Duplicados FMR + PO:",
    df_fmr_po.duplicated(
        subset=["FMR", "PO"]
    ).sum()
)

(31, 2)
FMR únicas: 20
PO únicas: 31
Duplicados FMR + PO: 0


8. NUEVO ANALISIS FMR --> PO

In [45]:
po_por_fmr = (
    df_fmr_po
    .groupby("FMR")["PO"]
    .nunique()
    .sort_values(ascending=False)
)

print(po_por_fmr)

FMR
748849    5
747667    3
762079    3
724411    2
739023    2
748204    2
290863    1
292230    1
393928    1
728366    1
750959    1
730855    1
758825    1
763898    1
777905    1
780263    1
783394    1
783399    1
786400    1
789066    1
Name: PO, dtype: int64


In [46]:
# Cuantas FMR están asociadas a cada PO ?
fmr_por_po = (
    df_fmr_po
    .groupby("PO")["FMR"]
    .nunique()
    .sort_values(ascending=False)
)

print(fmr_por_po)

PO
372068    1
372529    1
549934    1
734498    1
749089    1
768319    1
768320    1
770196    1
773448    1
773449    1
775430    1
775431    1
778686    1
778687    1
782496    1
783308    1
783376    1
783377    1
784422    1
786062    1
786063    1
786064    1
790214    1
791926    1
792338    1
794129    1
796220    1
796237    1
796255    1
798745    1
801426    1
Name: FMR, dtype: int64


In [47]:
# CUANTAS FMR REALES TINEN ALMENOS UNA PO
fmr_con_po = df_fmr_po["FMR"].nunique()

print("FMR con al menos una PO:", fmr_con_po)

FMR con al menos una PO: 20


In [48]:
# CUANTAS FMR SIN PO
fmr_sin_po = (
    set(df["FMR"].dropna().astype(str).str.strip())
    - set(df_fmr_po["FMR"].dropna().astype(str).str.strip())
)

print("FMR sin PO:")
print(fmr_sin_po)

FMR sin PO:
{'791970', '750147', '748236', '787733', '797282', '766793', '787641', '791795', '785312', '779569', 'Express N°22?', '788286', '781845', '787742', '…..................'}


In [49]:
print(df_fmr_po.shape)
print("FMR únicas:", df_fmr_po["FMR"].nunique())
print("PO únicas:", df_fmr_po["PO"].nunique())

print(
    "Duplicados FMR + PO:",
    df_fmr_po.duplicated(
        subset=["FMR", "PO"]
    ).sum()
)

print("\nPO por FMR:")
print(po_por_fmr)

print("\nFMR por PO:")
print(fmr_por_po)

print("\nFMR sin PO:")
print(fmr_sin_po)

(31, 2)
FMR únicas: 20
PO únicas: 31
Duplicados FMR + PO: 0

PO por FMR:
FMR
748849    5
747667    3
762079    3
724411    2
739023    2
748204    2
290863    1
292230    1
393928    1
728366    1
750959    1
730855    1
758825    1
763898    1
777905    1
780263    1
783394    1
783399    1
786400    1
789066    1
Name: PO, dtype: int64

FMR por PO:
PO
372068    1
372529    1
549934    1
734498    1
749089    1
768319    1
768320    1
770196    1
773448    1
773449    1
775430    1
775431    1
778686    1
778687    1
782496    1
783308    1
783376    1
783377    1
784422    1
786062    1
786063    1
786064    1
790214    1
791926    1
792338    1
794129    1
796220    1
796237    1
796255    1
798745    1
801426    1
Name: FMR, dtype: int64

FMR sin PO:
{'791970', '750147', '748236', '787733', '797282', '766793', '787641', '791795', '785312', '779569', 'Express N°22?', '788286', '781845', '787742', '…..................'}


9. ANALISIS FINAL DE FMR

In [50]:
fmr_status = (
    df[
        df["FMR"].notna()
        & ~df["FMR"].astype(str).str.contains("…", regex=False, na=False)
    ][["FMR", "STATUS"]]
    .drop_duplicates()
)

print(fmr_status)

              FMR                        STATUS
0          739023                    k. Awarded
5          739023    f. Bid Tab Issued to MYSRL
12         748849                    k. Awarded
15         748849                 b. RFQ Issued
20         748849  h. FPO to Supplier Signature
23         758825                    k. Awarded
24         762079                    k. Awarded
27         766793              c. TEA Requested
29         777905                    k. Awarded
30         789066                    k. Awarded
31         724411                    k. Awarded
34         724411                 b. RFQ Issued
45         786400                    k. Awarded
46         787641                 b. RFQ Issued
47         787641              c. TEA Requested
49         787742              c. TEA Requested
50         787733                 b. RFQ Issued
51         791970                 b. RFQ Issued
52         730855                           NaN
53         791795                       

In [51]:
print(
    fmr_status["STATUS"]
    .value_counts(dropna=False)
)

STATUS
k. Awarded                      17
NaN                             10
b. RFQ Issued                    5
c. TEA Requested                 4
f. Bid Tab Issued to MYSRL       2
h. FPO to Supplier Signature     2
a. FMR Received                  1
In Process                       1
Name: count, dtype: int64
